# Recipe 01 — Tracing
> Instrument your pipeline with traces and spans for end-to-end visibility.

| | |
|---|---|
| **Module** | `anchor.observability` |
| **Key classes** | `Tracer`, `Span`, `TraceRecord`, `SpanKind` |
| **Difficulty** | Beginner |

In [ ]:
# Setup
from anchor.observability import Tracer, Span, TraceRecord, SpanKind
import time

In [ ]:
# Create a tracer — entry point for all tracing
tracer = Tracer()
print(f"Tracer ready: {type(tracer).__name__}")

## 1 — Start a trace and add spans

In [ ]:
# A trace groups related spans under a single root identifier
trace = tracer.start_trace("my-pipeline", attributes={"pipeline": "rag"})
print(f"Trace ID   : {trace.trace_id}")
print(f"Trace name : my-pipeline")
print(f"Attributes : {trace.attributes}")

In [ ]:
# Retrieval span — use SpanKind to label each operation
span1 = tracer.start_span(
    trace.trace_id,
    "retrieval",
    kind=SpanKind.RETRIEVAL,
    attributes={"top_k": 10},
)
time.sleep(0.01)  # Simulate retrieval work
tracer.end_span(span1)
print(f"Span 1: {span1.name} (kind={span1.kind})")

In [ ]:
# Reranking span — child of retrieval via parent_span_id
span2 = tracer.start_span(
    trace.trace_id,
    "reranking",
    kind=SpanKind.RERANKING,
    parent_span_id=span1.span_id,
)
time.sleep(0.01)  # Simulate reranking
tracer.end_span(span2)
print(f"Span 2: {span2.name} (parent={span1.span_id[:8]}...)")

## 2 — End the trace and inspect results

In [ ]:
trace = tracer.end_trace(trace)

print(f"Trace   : {trace.trace_id}")
print(f"Spans   : {len(trace.spans)}")
print(f"Duration: {trace.total_duration_ms:.1f}ms")

In [ ]:
# Walk the span tree — useful for flame graphs or debugging
for span in trace.spans:
    parent = span.parent_span_id[:8] if span.parent_span_id else "root"
    print(f"  {span.name:12s}  kind={str(span.kind):12s}  parent={parent}")

## 3 — Multiple concurrent traces

In [ ]:
# A single Tracer can manage many traces at once
t1 = tracer.start_trace("pipeline-a", attributes={"model": "haiku"})
t2 = tracer.start_trace("pipeline-b", attributes={"model": "sonnet"})

s1 = tracer.start_span(t1.trace_id, "embed", kind=SpanKind.RETRIEVAL)
time.sleep(0.005)
tracer.end_span(s1)
t1 = tracer.end_trace(t1)

s2 = tracer.start_span(t2.trace_id, "generate", kind=SpanKind.RERANKING)
time.sleep(0.005)
tracer.end_span(s2)
t2 = tracer.end_trace(t2)

print(f"Trace A: {t1.total_duration_ms:.1f}ms  spans={len(t1.spans)}")
print(f"Trace B: {t2.total_duration_ms:.1f}ms  spans={len(t2.spans)}")

In [ ]:
# Inspect attributes on each trace
for t in [t1, t2]:
    print(f"  {t.trace_id[:8]}...  attrs={t.attributes}  "
          f"spans={len(t.spans)}  dur={t.total_duration_ms:.1f}ms")

In [ ]:
# Verify span kinds are preserved
for t in [t1, t2]:
    for sp in t.spans:
        print(f"  [{t.trace_id[:8]}] {sp.name}  kind={sp.kind}")

## Key Takeaways
- `Tracer` manages the lifecycle of traces and spans.
- `SpanKind` labels each operation (RETRIEVAL, RERANKING, etc.).
- Spans can be nested via `parent_span_id` to form a tree.
- `trace.total_duration_ms` gives end-to-end latency.

**Next:** [Exporters](02_exporters.ipynb)